## Homework 2 Additional Boosting

In [ ]:
# Required imports
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, balanced_accuracy_score, classification_report

# LightGBM
from lightgbm import LGBMClassifier

# CatBoost
from catboost import CatBoostClassifier

In [ ]:
train_df = pd.read_csv("playground-series-s6e4/train.csv")
test_df = pd.read_csv("playground-series-s6e4/test.csv")

print("Train shape:", train_df.shape)
print("Test shape:", test_df.shape)

# Set target column explicitly for this competition
target_col = "Irrigation_Need"
id_col = "id" if "id" in train_df.columns else None

print("Target column:", target_col)
print("ID column:", id_col)
print("Target values:", train_df[target_col].unique())

In [ ]:
feature_cols = [c for c in train_df.columns if c not in [target_col, id_col]] if id_col else [c for c in train_df.columns if c != target_col]

X = train_df[feature_cols].copy()
y = train_df[target_col].copy()
X_test = test_df[feature_cols].copy()

X_train, X_valid, y_train, y_valid = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("X_train:", X_train.shape)
print("X_valid:", X_valid.shape)
print("Target in X_train?", target_col in X_train.columns)
print("y_train dtype:", y_train.dtype)
print("Unique target values:", y_train.unique())

# Encode target labels for models that require numeric classes
label_encoder = LabelEncoder()
y_train_enc = label_encoder.fit_transform(y_train)
y_valid_enc = label_encoder.transform(y_valid)

print("Encoded classes:", list(label_encoder.classes_))

In [ ]:
# LightGBM baseline
# One-hot encode categoricals for a simple baseline
X_train_lgb = pd.get_dummies(X_train, drop_first=False)
X_valid_lgb = pd.get_dummies(X_valid, drop_first=False)
X_test_lgb = pd.get_dummies(X_test, drop_first=False)

# Align columns after one-hot encoding
X_train_lgb, X_valid_lgb = X_train_lgb.align(X_valid_lgb, join="left", axis=1, fill_value=0)
X_train_lgb, X_test_lgb = X_train_lgb.align(X_test_lgb, join="left", axis=1, fill_value=0)

lgb_model = LGBMClassifier(
    objective="multiclass",
    num_class=len(label_encoder.classes_),
    n_estimators=500,
    learning_rate=0.05,
    num_leaves=31,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
)

lgb_model.fit(X_train_lgb, y_train_enc)

lgb_valid_preds = lgb_model.predict(X_valid_lgb)

lgb_acc = accuracy_score(y_valid_enc, lgb_valid_preds)
lgb_bal_acc = balanced_accuracy_score(y_valid_enc, lgb_valid_preds)

print(f"LightGBM Accuracy: {lgb_acc:.6f}")
print(f"LightGBM Balanced Accuracy: {lgb_bal_acc:.6f}")
print(classification_report(y_valid_enc, lgb_valid_preds, target_names=label_encoder.classes_))

In [ ]:
# CatBoost baseline
# CatBoost can directly use categorical columns
cat_features = X_train.select_dtypes(include=["object", "category"]).columns.tolist()
cat_feature_indices = [X_train.columns.get_loc(col) for col in cat_features]

cat_model = CatBoostClassifier(
    iterations=500,
    learning_rate=0.05,
    depth=6,
    loss_function="MultiClass",
    eval_metric="Accuracy",
    random_seed=42,
    verbose=0,
)

cat_model.fit(
    X_train,
    y_train_enc,
    cat_features=cat_feature_indices,
    eval_set=(X_valid, y_valid_enc),
    use_best_model=True,
)

cat_valid_preds = cat_model.predict(X_valid)
cat_valid_preds = cat_valid_preds.flatten().astype(int)

cat_acc = accuracy_score(y_valid_enc, cat_valid_preds)
cat_bal_acc = balanced_accuracy_score(y_valid_enc, cat_valid_preds)

print(f"CatBoost Accuracy: {cat_acc:.6f}")
print(f"CatBoost Balanced Accuracy: {cat_bal_acc:.6f}")
print(classification_report(y_valid_enc, cat_valid_preds, target_names=label_encoder.classes_))

In [ ]:
param_grid_1 = {
    "learning_rate": 0.05,
    "n_estimators": 500,
    "num_leaves": 31,
    "max_depth": -1,
    "min_child_samples": 20,
    "subsample": 0.8,
    "colsample_bytree": 0.8,
    "random_state": 42
}

param_grid_2 = {
    "learning_rate": 0.03,
    "n_estimators": 300,
    "num_leaves": 63,
    "max_depth": -1,
    "min_child_samples": 20,
    "random_state": 42
}

param_grid_3 = {
    "learning_rate": 0.02,
    "n_estimators": 500,
    "num_leaves": 31,
    "max_depth": -1,
    "min_child_samples": 20,
    "random_state": 42
}

param_grids = {
    "LightGBM_Model_1": param_grid_1,
    "LightGBM_Model_2": param_grid_2,
    "LightGBM_Model_3": param_grid_3
}

In [ ]:
lgb_results = []
lgb_models = {}

for model_name, params in param_grids.items():
    model = LGBMClassifier(
        objective="multiclass",
        num_class=len(label_encoder.classes_),
        **params
    )
    
    model.fit(X_train_lgb, y_train_enc)
    preds = model.predict(X_valid_lgb)
    
    acc = accuracy_score(y_valid_enc, preds)
    bal_acc = balanced_accuracy_score(y_valid_enc, preds)
    
    lgb_results.append({
        "Model": model_name,
        "learning_rate": params.get("learning_rate"),
        "n_estimators": params.get("n_estimators"),
        "num_leaves": params.get("num_leaves"),
        "max_depth": params.get("max_depth"),
        "min_child_samples": params.get("min_child_samples"),
        "subsample": params.get("subsample", 1.0),
        "colsample_bytree": params.get("colsample_bytree", 1.0),
        "Validation Accuracy": acc,
        "Validation Balanced Accuracy": bal_acc
    })
    
    lgb_models[model_name] = model
    
    print(f"\n{model_name}")
    print("-" * 50)
    print(f"Accuracy: {acc:.6f}")
    print(f"Balanced Accuracy: {bal_acc:.6f}")
    print(classification_report(y_valid_enc, preds, target_names=label_encoder.classes_))

In [ ]:
lgb_results_df = pd.DataFrame(lgb_results).sort_values(
    by="Validation Balanced Accuracy",
    ascending=False
).reset_index(drop=True)

lgb_results_df

In [ ]:
best_lgb_test_preds = best_lgb_model.predict(X_test_lgb)
best_lgb_test_labels = label_encoder.inverse_transform(best_lgb_test_preds.astype(int))

submission_lgb = pd.DataFrame({
    "id": test_df["id"],
    "Irrigation_Need": best_lgb_test_labels
})

submission_lgb.to_csv("Output/best_lightgbm_submission.csv", index=False)
submission_lgb.head()

In [ ]:
cat_features = X_train.select_dtypes(include=["object", "category"]).columns.tolist()
cat_feature_indices = [X_train.columns.get_loc(col) for col in cat_features]

cat_model = CatBoostClassifier(
    iterations=500,
    learning_rate=0.05,
    depth=6,
    loss_function="MultiClass",
    eval_metric="Accuracy",
    random_seed=42,
    verbose=0
)

cat_model.fit(
    X_train,
    y_train_enc,
    cat_features=cat_feature_indices,
    eval_set=(X_valid, y_valid_enc),
    use_best_model=True
)

cat_valid_preds = cat_model.predict(X_valid)
cat_valid_preds = cat_valid_preds.flatten().astype(int)

cat_acc = accuracy_score(y_valid_enc, cat_valid_preds)
cat_bal_acc = balanced_accuracy_score(y_valid_enc, cat_valid_preds)

print(f"CatBoost Accuracy: {cat_acc:.6f}")
print(f"CatBoost Balanced Accuracy: {cat_bal_acc:.6f}")
print(classification_report(y_valid_enc, cat_valid_preds, target_names=label_encoder.classes_))

In [ ]:
cat_test_preds = cat_model.predict(X_test)
cat_test_preds = cat_test_preds.flatten().astype(int)
cat_test_labels = label_encoder.inverse_transform(cat_test_preds)

submission_cat = pd.DataFrame({
    "id": test_df["id"],
    "Irrigation_Need": cat_test_labels
})

submission_cat.to_csv("Output/catboost_submission.csv", index=False)
submission_cat.head()

In [ ]:
# New LightGBM model with updated hyperparameters

from lightgbm import LGBMClassifier
from sklearn.metrics import accuracy_score, balanced_accuracy_score, classification_report

param_grid_new = {
    "learning_rate": 0.05,
    "n_estimators": 500,
    "num_leaves": 63,
    "max_depth": -1,
    "min_child_samples": 20,
    "subsample": 0.8,
    "colsample_bytree": 0.8,
    "random_state": 42
}

lgb_model_new = LGBMClassifier(
    objective="multiclass",
    num_class=len(label_encoder.classes_),
    **param_grid_new
)

# Train
lgb_model_new.fit(X_train_lgb, y_train_enc)

# Predict
lgb_valid_preds_new = lgb_model_new.predict(X_valid_lgb)

# Evaluate
acc_new = accuracy_score(y_valid_enc, lgb_valid_preds_new)
bal_acc_new = balanced_accuracy_score(y_valid_enc, lgb_valid_preds_new)

print("New LightGBM Model (num_leaves=63)")
print("-" * 50)
print(f"Accuracy: {acc_new:.6f}")
print(f"Balanced Accuracy: {bal_acc_new:.6f}")
print(classification_report(y_valid_enc, lgb_valid_preds_new, target_names=label_encoder.classes_))

In [ ]:
test_preds_new = lgb_model_new.predict(X_test_lgb)
test_labels_new = label_encoder.inverse_transform(test_preds_new.astype(int))

submission_new = pd.DataFrame({
    "id": test_df["id"],
    "Irrigation_Need": test_labels_new
})

submission_new.to_csv("Output/lightgbm_model_num_leaves_63.csv", index=False)
submission_new.head()

In [ ]:
import optuna
from lightgbm import LGBMClassifier
from sklearn.metrics import balanced_accuracy_score

def objective(trial):
    
    params = {
        "objective": "multiclass",
        "num_class": len(label_encoder.classes_),
        
        # Hyperparameters to tune
        "n_estimators": trial.suggest_int("n_estimators", 100, 500),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.1),
        "num_leaves": trial.suggest_int("num_leaves", 10, 100),
        "min_child_samples": trial.suggest_int("min_child_samples", 10, 50),
        
        "subsample": trial.suggest_float("subsample", 0.7, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.7, 1.0),
        
        "random_state": 42
    }

    model = LGBMClassifier(**params)
    
    model.fit(X_train_lgb, y_train_enc)
    
    preds = model.predict(X_valid_lgb)
    
    score = balanced_accuracy_score(y_valid_enc, preds)
    
    return score

In [ ]:
study = optuna.create_study(direction="maximize")

study.optimize(objective, n_trials=20)  # you can increase to 50–100 later

In [ ]:
print("Best trial score:", study.best_value)
print("Best parameters:")
print(study.best_params)

In [ ]:
best_params = study.best_params

best_model_optuna = LGBMClassifier(
    objective="multiclass",
    num_class=len(label_encoder.classes_),
    random_state=42,
    **best_params
)

best_model_optuna.fit(X_train_lgb, y_train_enc)

optuna_preds = best_model_optuna.predict(X_valid_lgb)

from sklearn.metrics import accuracy_score

print("Optuna Accuracy:", accuracy_score(y_valid_enc, optuna_preds))
print("Optuna Balanced Accuracy:", balanced_accuracy_score(y_valid_enc, optuna_preds))

In [ ]:
test_preds_optuna = best_model_optuna.predict(X_test_lgb)
test_labels_optuna = label_encoder.inverse_transform(test_preds_optuna.astype(int))

submission_optuna = pd.DataFrame({
    "id": test_df["id"],
    "Irrigation_Need": test_labels_optuna
})

submission_optuna.to_csv("Output/optuna_lightgbm_submission.csv", index=False)
submission_optuna.head()